# Octave · Kriging vs STK

**Idea.** The Octave/MATLAB reference toolbox, STK: at the same hyperparameters
(Gaussian kernel) both must give the same predictor, and their default fits are
compared.

**Reference.** STK (exact equality, then accuracy and cost).

**Tags.** class: Kriging · kernel: gauss, matern5_2 · compared to: STK

[STK](https://github.com/stk-kriging/stk) (Small Toolbox for Kriging) is the
reference kriging toolbox for Octave/MATLAB.

1. **Same hyperparameters → same numbers** (error-level). STK's anisotropic
   Gaussian covariance is σ² exp(−‖(x−x')/ρ‖²) and libKriging's `gauss` is
   σ² exp(−½ Σ ((xⱼ−x'ⱼ)/θⱼ)²): the same kernel with ρ = θ √2. With a constant
   mean (ordinary/universal kriging, β estimated by GLS in both), the
   predicted mean and variance at fixed (θ, σ²) must coincide. (STK's Matérn
   covariances are radial, libKriging's are product-form: not comparable
   hyperparameter by hyperparameter, hence the Gaussian kernel here.)
2. **Default fits** on the shared datasets of `data/`: Matérn 5/2, each
   toolbox's own estimation (warn-level comparison of Q²).

mLibKriging must be on the path (`OCTAVE_PATH` or `addpath`), see
`tools/install/octave.sh`.

In [1]:
source("tools/octave/gallery.m");
s = suite("vs-STK_Kriging_octave");

## Installing STK

STK is fetched here (shallow clone of its repository into a temporary
directory), unless the `STK_PATH` environment variable already points to a copy.
Without STK, the comparison is skipped (warn-level).

In [2]:
stk_path = getenv("STK_PATH");
if isempty(stk_path) || ~exist(fullfile(stk_path, "stk_init.m"), "file")
  stk_path = fullfile(tempdir(), "stk");
  if ~exist(fullfile(stk_path, "stk_init.m"), "file")
    [status, out] = system(sprintf("git clone --depth 1 https://github.com/stk-kriging/stk.git %s 2>&1", stk_path));
    printf("git clone stk: status %d\n%s", status, out);
  end
end
have_stk = exist(fullfile(stk_path, "stk_init.m"), "file") == 2;
if have_stk
  run(fullfile(stk_path, "stk_init.m"));
  printf("STK %s from %s\n", stk_version(), stk_path);
else
  s = check(s, "STK available", false, "absent", stk_path, "STK could not be installed: comparison skipped", "warn");
end
datasets = {"branin", "hartmann3", "borehole"};

git clone stk: status 0
Cloning into '/tmp/stk'...


Compiling MEX-file stk_dist_matrixx... ok.


Compiling MEX-file stk_dist_matrixy... ok.


Compiling MEX-file stk_dist_pairwise... ok.


Compiling MEX-file stk_filldist_discr_mex... ok.

...


Compiling MEX-file stk_sampling_vdc_rr2... ok.


Compiling MEX-file stk_sampling_sobol_mex... ok.


Compiling MEX-file stk_paretofind_mex... ok.


Compiling MEX-file stk_isdominated_mex... ok.


Compiling MEX-file stk_dominatedhv_mex... ok.
STK 2.8.0 from /tmp/stk


## 1. Same hyperparameters, same numbers

In [3]:
if have_stk
  for i = 1:numel(datasets)
    name = datasets{i};
    X = read_data(name, "X"); y = read_data(name, "y"); Xt = read_data(name, "Xt");
    d = columns(X);
    theta = linspace(0.2, 0.6, d);
    k = Kriging(y, X, "gauss", "constant", false, "none", "LL", Params("theta", theta, "is_theta_estim", false));
    sigma2 = k.sigma2();
    [m_lk, s_lk] = k.predict(Xt, true, false, false);
    model = stk_model(@stk_gausscov_aniso, d);
    model.param = [log(sigma2); -log(theta(:) * sqrt(2))];
    model.lm = stk_lm_constant();
    zp = stk_predict(model, X, y, Xt);
    s = check_close(s, [name ": gauss UK mean == STK"], m_lk, zp.mean, 1e-6, 1e-8 * std(y));
    s = check_close(s, [name ": gauss UK variance == STK"], s_lk.^2, zp.var, 1e-4, 1e-8 * sigma2);
  end
end

## 2. Default fits

In [4]:
for i = 1:numel(datasets)
  name = datasets{i};
  X = read_data(name, "X"); y = read_data(name, "y"); Xt = read_data(name, "Xt"); yt = read_data(name, "yt");
  d = columns(X);
  tic; k = Kriging(y, X, "matern5_2"); t = toc;
  [m, ~] = k.predict(Xt, false, false, false);
  q_lk = q2(yt, m);
  s = metric(s, name, "mLibKriging", "q2", q_lk); s = metric(s, name, "mLibKriging", "fit_s", t);
  tic; k10 = Kriging(y, X, "matern5_2", "constant", false, "BFGS10", "LL"); t = toc;
  [m, ~] = k10.predict(Xt, false, false, false);
  q_lk10 = q2(yt, m);
  s = metric(s, name, "mLibKriging BFGS10", "q2", q_lk10); s = metric(s, name, "mLibKriging BFGS10", "fit_s", t);
  s = check(s, [name ": Q2(mLibKriging BFGS10) > 0.9"], q_lk10 > 0.9, q_lk10, "> 0.9");
  if have_stk
    tic;
    model = stk_model(@stk_materncov52_aniso, d);
    model = stk_param_estim(model, X, y);  % (STK >= 2.8 returns the model)
    zp = stk_predict(model, X, y, Xt);
    t = toc;
    q_stk = q2(yt, zp.mean);
    s = metric(s, name, "STK", "q2", q_stk); s = metric(s, name, "STK", "fit_s", t);
    s = check(s, [name ": Q2(mLibKriging) - Q2(STK)"], q_lk - q_stk > -0.02, q_lk - q_stk, "> -0.02", "", "warn");
  end
end
disp(s.metrics(:, 3:6))

{
  [1,1] = branin
  [2,1] = branin
  [3,1] = branin
  [4,1] = branin
  [5,1] = branin
  [6,1] = branin
  [7,1] = hartmann3
  [8,1] = hartmann3
  [9,1] = hartmann3
  [10,1] = hartmann3
  [11,1] = hartmann3
  [12,1] = hartmann3
  [13,1] = borehole
  [14,1] = borehole
  [15,1] = borehole
  [16,1] = borehole
  [17,1] = borehole
  [18,1] = borehole
  [1,2] = mLibKriging
  [2,2] = mLibKriging
  [3,2] = mLibKriging BFGS10
  [4,2] = mLibKriging BFGS10
  [5,2] = STK
  [6,2] = STK
  [7,2] = mLibKriging
  [8,2] = mLibKriging
  [9,2] = mLibKriging BFGS10
  [10,2] = mLibKriging BFGS10
  [11,2] = STK
  [12,2] = STK
  [13,2] = mLibKriging
  [14,2] = mLibKriging
  [15,2] = mLibKriging BFGS10
  [16,2] = mLibKriging BFGS10
  [17,2] = STK
  [18,2] = STK
  [1,3] = q2
  [2,3] = fit_s
  [3,3] = q2
  [4,3] = fit_s
  [5,3] = q2
  [6,3] = fit_s
  [7,3] = q2
  [8,3] = fit_s
  [9,3] = q2
  [10,3] = fit_s
  [11,3] = q2
  [12,3] = fit_s
  [13,3] = q2
  [14,3] = fit_s
  [15,3] = q2
  [16,3] = fit_s
  [17,3] = q2
 

## Summary

The metrics of this notebook, the number of checks per status, and every check that
did not pass (FAIL: error-level, fails the run; WARN: warn-level; KNOWN: listed in
`tools/known_issues.csv`, with its issue; SKIP: feature absent from the installed version).
All of them are also written to `results/vs-STK_Kriging_octave.checks.csv` and `.metrics.csv`.

In [5]:
suite_summary(s);

branin                       mLibKriging                  q2           0.999036
branin                       mLibKriging                  fit_s        0.000998974
branin                       mLibKriging BFGS10           q2           0.999036
branin                       mLibKriging BFGS10           fit_s        0.00749111
branin                       STK                          q2           0.997945
branin                       STK                          fit_s        0.148749
hartmann3                    mLibKriging                  q2           0.976728
hartmann3                    mLibKriging                  fit_s        0.00511885
hartmann3                    mLibKriging BFGS10           q2           0.976728
hartmann3                    mLibKriging BFGS10           fit_s        0.0556099
hartmann3                    STK                          q2           0.9761
hartmann3                    STK                          fit_s        0.105487
borehole                     mLibK

In [6]:
suite_finalize(s);

vs-STK_Kriging_octave: PASS 12, FAIL 0, WARN 0, KNOWN 0, SKIP 0
